# Create IDs for your data

Files need to be stored in .nii.gz format.


In [ ]:
import os
import random
import pandas as pd

#Set random seed for reproducibility
random.seed(42)

def get_image_paths(base_folder):
    """Get a list of all .nii.gz file paths in the given folder (including subfolders if they exist)."""
    image_paths = []
    
    for root, _, files in os.walk(base_folder):
        for filename in files:
            if filename.endswith('.nii.gz'):
                image_paths.append(os.path.join(root, filename))
    
    return image_paths

def split_data(image_paths, train_size, val_size):
    """Randomly split the dataset into training and validation sets."""

    random.shuffle(image_paths)
    train_paths = image_paths[:train_size]
    val_paths = image_paths[train_size:train_size + val_size]
    
    return train_paths, val_paths

def save_to_csv(file_paths, output_csv):
    """Save file paths to a CSV file."""
    df = pd.DataFrame(file_paths, columns=["image"])
    df.to_csv(output_csv, index=False)

def main(base_folder, train_csv, val_csv, train_size=300, val_size=60):
    """Main function to split data and save the paths to CSV files."""

    image_paths = get_image_paths(base_folder)
    total_images = len(image_paths)
    if total_images < train_size + val_size:
        print(f"Error: Not enough images. Found {total_images}, but need {train_size + val_size}.")
        return

    train_paths, val_paths = split_data(image_paths, train_size, val_size)

    save_to_csv(train_paths, train_csv)
    save_to_csv(val_paths, val_csv)
    
    print(f"Training data saved to {train_csv}")
    print(f"Validation data saved to {val_csv}")


# Define the base folder containing the subfolders with CT volumes
base_folder = './data'  # Replace with your folder path

# Define the paths to save the CSV files
train_csv = './ids/train.csv'
val_csv = './ids/validation.csv'

main(base_folder, train_csv, val_csv)


# Visualize Image Chracteristics

1) Image Dimensions

In [ ]:
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

def load_image_dimensions_and_intensity(file_path):
    """Load image and return its dimensions (shape) and intensity values."""
    file_path=file_path.replace('sds-hd','sds')
    img = nib.load(file_path)
    return img.shape

def print_dimension_and_intensity_statistics(csv_file):
    """Print statistics of the dimensions and intensity distributions of the .nii.gz files listed in the CSV and show histograms."""

    df = pd.read_csv(csv_file, sep=",")

    dimensions = []
    

    for image_path in df['image']:
        try:
            dims = load_image_dimensions_and_intensity(image_path)
            dimensions.append(dims)
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
    
    dims_df = pd.DataFrame(dimensions, columns=["Width", "Height", "Depth"])
    
    # Print statistics for each dimension
    print("Dimension Statistics:")
    print(f"Total number of images: {len(dimensions)}")
    print("Depth (z) statistics:")
    print(dims_df['Depth'].describe())
    print("Width (x) statistics:")
    print(dims_df['Width'].describe())
    print("Height (y) statistics:")
    print(dims_df['Height'].describe())


    dims_df.hist(column=["Depth", "Width", "Height"], bins=50, figsize=(12, 8), layout=(2, 2))
    plt.suptitle("Histograms of Image Dimensions")
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to fit the title
    plt.show()
    
print("Statistics for training images:")
print_dimension_and_intensity_statistics("./train.csv")  # Replace with your actual file path if necessary
print("\nStatistics for testing images:")
print_dimension_and_intensity_statistics("./validation.csv")  # Replace with your actual file path if necessary


2) Image Spacing

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import json

def get_spacing_from_dataloader(dataloader):
    spacings = []
    for batch in dataloader:
        meta = batch["image"].meta
        affine = batch["image"].affine.squeeze(0)
        spacing = torch.tensor([torch.norm(affine[0, :3]),
                        torch.norm(affine[1, :3]),
                        torch.norm(affine[2, :3])])

        spacing_og = meta["pixdim"][0][1:4].cpu().numpy()  # Ensure it's converted to NumPy array
        spacings.append(spacing.cpu().numpy())

    return np.array(spacings)

def get_spacing_from_folder(folder_path):
    spacings = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".nii.gz"):
            filepath = os.path.join(folder_path, filename)
            img = nib.load(filepath)  # Load the NIfTI file
            spacing = img.header.get_zooms()  # Extract spacing
            spacings.append(spacing)
    return np.array(spacings)

def get_spacing_from_json(folder_path):
    spacings = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, "r") as f:
                data = json.load(f)
                if "spacing" in data:
                    spacings.append(data["spacing"])
    return np.array(spacings)

def analyze_spacing(spacings):
    # Compute statistics
    mean_spacing = np.mean(spacings, axis=0)
    median_spacing = np.median(spacings, axis=0)
    percentiles = np.percentile(spacings, [25, 50, 75], axis=0)
    
    # Print statistics
    print(f"Mean Spacing: {mean_spacing}")
    print(f"Median Spacing: {median_spacing}")
    print(f"25th, 50th, 75th Percentiles: {percentiles}")
    
    # Plot distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes_titles = ["Spacing in X", "Spacing in Y", "Spacing in Z"]
    
    for i in range(3):
        sns.histplot(spacings[:, i], bins=20, kde=True, ax=axes[i])
        axes[i].set_title(axes_titles[i])
        axes[i].set_xlabel("Spacing (mm)")
        axes[i].set_ylabel("Frequency")
    
    plt.tight_layout()
    plt.show()

# Main execution
def main(dataloader=None, folder_path=None, json_folder_path=None):
    if dataloader:
        spacings = get_spacing_from_dataloader(dataloader)
    elif folder_path:
        spacings = get_spacing_from_folder(folder_path)
    elif json_folder_path:
        spacings = get_spacing_from_json(json_folder_path)
    else:
        raise ValueError("Either a dataloader or a folder_path must be provided.")
    
    analyze_spacing(spacings)

# Example usage:
#main(dataloader=val_loader)
#main(json_folder_path="json_path")  # If using JSON files  # If using MONAI DataLoader
#main(folder_path="/path/to/your/data")  # If using folder


# Visualize Data

In [ ]:
import nibabel as nib
import numpy as np
import imageio
import matplotlib.pyplot as plt
from pathlib import Path

nii_file = "./image.nii.gz"  # Change this to your file path
img = nib.load(nii_file)

data = img.get_fdata()

# Normalize to [0, 255] for saving as images
data = (data - np.min(data)) / (np.max(data) - np.min(data)) * 255
data = data.astype(np.uint8)

# Define output paths
output_dir = Path("./output") 
output_dir.mkdir(exist_ok=True)  
output_gif = Path("./output/sample_1").with_suffix('.gif')

# Create a list of frames for the GIF
frames = []
for i in range(data.shape[2]):  # Loop through the depth (Z-axis)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(data[:, :, i], cmap="gray")  # Show the slice
    ax.axis('off')

    # Save the slice as a PNG
    slice_path = output_dir / f"slice_{i:03d}.png"
    plt.savefig(slice_path, bbox_inches='tight', pad_inches=0, dpi=100)
    print(f"Saved slice: {slice_path}")

    # Convert the plot to an RGB frame for GIF
    fig.canvas.draw()
    frame = np.array(fig.canvas.renderer.buffer_rgba())[:, :, :3]  # Convert to RGB
    frames.append(frame)

    plt.close(fig) 

# Save as GIF
imageio.mimsave(output_gif, frames, duration=0.1)  # Adjust duration if needed
print(f"GIF saved at: {output_gif}")
